# PuReMD: NaCl(aq) molecular dynamics (ReaxFF)

This notebook sets up and runs a **toy** reactive MD simulation of **aqueous NaCl** using **PuReMD**.

It is designed to be **similar in spirit** to your CHEM3580 workshop notebooks (parameter widgets, generate a system, run MD, then post-process), but uses **PuReMD** instead of OpenMM.

## What you need beforehand

1. A working **PuReMD** executable (CPU is fine for a workshop).
2. A **single ReaxFF parameter file** (`ffield.reax`) that supports at least:
   - `H`, `O`, `Na`, `Cl`
3. Basic build tools: `git`, a C/C++ compiler, `make` (and optionally MPI).

> ⚠️ ReaxFF parameter files are not “mix-and-match”: you must use **one unified `ffield`** that contains *all* elements in your simulation.

---

## Roadmap

1. Get PuReMD (clone + build)
2. Discover PuReMD input format from its own examples (no guessing)
3. Generate a simple NaCl(aq) starting structure (random packing with minimum distances)
4. Write PuReMD input files (geometry + control)
5. Run PuReMD
6. Quick sanity checks on outputs


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
import numpy as np

WORK = Path("puremd_nacl_water").resolve()
WORK.mkdir(exist_ok=True)
WORK


## 1) Get PuReMD (clone)

This will attempt to clone PuReMD into `puremd_nacl_water/PuReMD`.

If you already have PuReMD somewhere else (e.g. on Gadi), you can skip this cell and set `PUREMD_EXE` manually later.


In [ ]:
PUREMD_REPO = "https://github.com/msu-sparta/PuReMD.git"
PUREMD_DIR = WORK / "PuReMD"

if not PUREMD_DIR.exists():
    print("Cloning PuReMD...")
    subprocess.check_call(["git", "clone", "--depth", "1", "--recurse-submodules", PUREMD_REPO, str(PUREMD_DIR)])
else:
    print("PuReMD directory already exists:", PUREMD_DIR)

git_dir = PUREMD_DIR / ".git"
if git_dir.exists():
    print("Updating submodules...")
    subprocess.check_call(["git", "submodule", "update", "--init", "--recursive"], cwd=PUREMD_DIR)

# Show top-level files
print("\nTop-level files:")
for p in sorted(PUREMD_DIR.iterdir()):
    print(" -", p.name)


## 2) Build PuReMD

PuReMD has had multiple build systems across versions. This cell:
- looks for common build entry points,
- tries a simple build path,
- and asks you to confirm the resulting executable path.

If you’re on an HPC system, you may want to load modules first (e.g. `module load gcc openmpi`).


In [ ]:
import multiprocessing
import sys

def run(cmd, cwd=None):
    print("+", " ".join(str(x) for x in cmd))
    subprocess.check_call(cmd, cwd=cwd)

def is_colab():
    return "google.colab" in sys.modules

def colab_cuda_root():
    for candidate in [Path("/usr/local/cuda"), Path("/usr/local/cuda-12"), Path("/usr/local/cuda-11")]:
        if (candidate / "bin" / "nvcc").exists():
            return candidate
    return None

def find_executables(root: Path):
    hits = []
    skip_dirs = {".git", "build", "cmake-build-debug", "cmake-build-release", "__pycache__"}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        for name in filenames:
            lower = name.lower()
            if lower.endswith((".c", ".cc", ".cpp", ".h", ".hpp", ".o", ".a", ".so", ".dylib", ".txt", ".md", ".in", ".dat", ".xyz", ".py")):
                continue
            if lower in {"puremd", "puredmd", "puredmd-omp", "puredmd_gpu", "puredmd-gpu"} or name.startswith("PuReMD") or lower.startswith("puremd"):
                path = Path(dirpath) / name
                if path.is_file() and os.access(path, os.X_OK):
                    hits.append(path.resolve())
    return sorted(set(hits))

if is_colab():
    print("Detected Google Colab; installing GPU build dependencies...")
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", "autoconf", "automake", "build-essential", "cmake", "git", "libtool", "openmpi-bin", "libopenmpi-dev", "pkg-config", "zlib1g-dev"])
    cuda_root = colab_cuda_root()
    assert cuda_root is not None, "Could not find nvcc in Colab. Enable a GPU runtime first: Runtime -> Change runtime type -> T4/L4/A100 GPU."
    os.environ["PATH"] = f"{cuda_root / 'bin'}:{os.environ['PATH']}"
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_root / 'lib64'}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    run(["nvidia-smi"])
    run(["nvcc", "--version"])
else:
    cuda_root = None

exe_candidates = find_executables(PUREMD_DIR)
jobs = str(max(2, multiprocessing.cpu_count()))

if not exe_candidates and (PUREMD_DIR / "configure.ac").exists():
    print("Detected autotools build system; bootstrapping with autoreconf...")
    run(["autoreconf", "-fi"], cwd=PUREMD_DIR)

if not exe_candidates and (PUREMD_DIR / "configure").exists():
    print("Running configure for GPU-enabled PuReMD...")
    run(["chmod", "+x", "configure"], cwd=PUREMD_DIR)
    configure_cmd = ["./configure"]
    if cuda_root is not None:
        configure_cmd.extend(["--enable-serial=no", "--enable-mpi-cuda=yes", f"--with-cuda={cuda_root}"])
    run(configure_cmd, cwd=PUREMD_DIR)

make_dirs = [d for d in [PUREMD_DIR, PUREMD_DIR / "src", PUREMD_DIR / "PuReMD", PUREMD_DIR / "PuReMD-OMP", PUREMD_DIR / "PuReMD-GPU"] if (d / "Makefile").exists()]
print("Makefile directories found:")
for d in make_dirs:
    print(" -", d)

assert make_dirs, "No Makefile found after bootstrapping. Inspect the repo contents printed above and build manually."

target_candidates = [None, "PuReMD-GPU", "GPU", "PuReMD", "puremd", "OMP", "serial", "cpu", "all"]

for make_dir in make_dirs:
    if exe_candidates:
        break
    print(f"\nTrying builds in {make_dir} ...")
    for target in target_candidates:
        cmd = ["make", f"-j{jobs}"]
        if target is not None:
            cmd.append(target)
        try:
            run(cmd, cwd=make_dir)
        except subprocess.CalledProcessError as e:
            label = "default target" if target is None else target
            print(f"Build failed for {label} in {make_dir}: return code {e.returncode}")
            continue
        exe_candidates = find_executables(PUREMD_DIR)
        if exe_candidates:
            print("Build produced executable(s).")
            break

print("\nExecutable candidates found:")
for p in exe_candidates[:50]:
    print(" -", p)

if not exe_candidates:
    raise RuntimeError("PuReMD did not build successfully. Scroll up to the failing configure/make command in this cell and inspect the CUDA or compiler error.")


In [ ]:
# --- Set this to the PuReMD executable you want to run ---
# If the build above found candidates, pick one.
PUREMD_EXE = None

# Prefer a CUDA/MPI executable when multiple variants were built.
if exe_candidates:
    preferred = sorted(
        exe_candidates,
        key=lambda p: (
            not any(tag in str(p).lower() for tag in ["cuda", "gpu", "mpi"]),
            len(str(p)),
            str(p),
        ),
    )
    PUREMD_EXE = str(preferred[0])

print("PUREMD_EXE =", PUREMD_EXE)
assert PUREMD_EXE is not None, "Set PUREMD_EXE to your PuReMD executable path."


## 3) Discover PuReMD input format from examples (recommended)

Rather than hard-coding assumptions about PuReMD’s `geo` and `control` formats, we will:
- search the cloned repo for example input files,
- open a couple,
- and then generate our own with the same structure.

This makes the notebook much more robust across PuReMD forks/versions.


In [ ]:
# Search for likely example input files
likely = []
for root, dirs, files in os.walk(PUREMD_DIR):
    for f in files:
        name = f.lower()
        if name in {"control", "control.in", "control.txt", "control.reax", "geo", "geo.in", "geometry", "input.geo"}:
            likely.append(Path(root) / f)

# Also include small text files that mention 'imdmet' or 'qeq'
for root, dirs, files in os.walk(PUREMD_DIR):
    for f in files:
        if f.lower().endswith((".in", ".txt", ".dat")):
            path = Path(root) / f
            try:
                txt = path.read_text(errors="ignore")
            except Exception:
                continue
            if ("imdmet" in txt.lower()) or ("qeq" in txt.lower()) or ("reaxff" in txt.lower() and "control" in path.name.lower()):
                likely.append(path)

likely = sorted(set(likely))
print(f"Found {len(likely)} candidate example/config files.")
for p in likely[:40]:
    print(" -", p.relative_to(PUREMD_DIR))


In [ ]:
# Peek at a few candidates so we can mimic their formatting
def head(path, n=60):
    txt = Path(path).read_text(errors="ignore").splitlines()
    return "\n".join(txt[:n])

# Choose up to 2 likely 'control' and 2 likely 'geo' files
controls = [p for p in likely if p.name.lower().startswith("control")]
geos = [p for p in likely if p.name.lower().startswith(("geo", "geometry", "input.geo"))]

print("CONTROL candidates:")
for p in controls[:5]:
    print("\n===", p.relative_to(PUREMD_DIR), "===\n")
    print(head(p, 60))

print("\n\nGEO candidates:")
for p in geos[:5]:
    print("\n===", p.relative_to(PUREMD_DIR), "===\n")
    print(head(p, 60))


## 4) Provide your ReaxFF parameter file (`ffield.reax`)

Place your parameter file in:

- `puremd_nacl_water/ffield.reax`

(or change `FFIELD_PATH` below)

Typical sources are supporting information (SI) from ReaxFF water/NaCl parameterization papers, or group-distributed parameter packs.

This notebook does not download parameter files for you.


In [ ]:
FFIELD_PATH = WORK / "ffield.reax"
print("Expecting:", FFIELD_PATH)

if not FFIELD_PATH.exists():
    print("\n⚠️ ffield.reax not found yet.")
    print("Copy your unified H/O/Na/Cl ReaxFF file to this path, then re-run this cell.")
else:
    # Show first few lines for a sanity check
    print("\nFound ffield.reax. Header:")
    print("\n".join(FFIELD_PATH.read_text(errors="ignore").splitlines()[:10]))


## 5) Build a simple NaCl(aq) starting structure

We will generate:
- `Nw` water molecules (as *three atoms*, O/H/H) with roughly correct geometry
- `Nion` Na⁺ and `Nion` Cl⁻ ions
- A cubic periodic box of length `L` Å

### Notes
- This is a *simple random packing* with minimum distance constraints, intended for **teaching/demo**.
- For serious work, use Packmol / moltemplate / pre-equilibrated boxes.


In [ ]:
# --- User-editable parameters (workshop-friendly) ---
T_K = 298.0
P_atm = 1.0

L_ang = 25.0         # box length (Å)
Nw = 200             # number of waters
Nion = 4             # number of Na and number of Cl (total ions = 2*Nion)

min_dist = 1.6       # Å, crude minimum interatomic separation in initial packing
seed = 7

rng = np.random.default_rng(seed)


In [ ]:
def random_points_in_box(n, L, rng):
    return rng.random((n, 3)) * L

def place_atoms_with_min_dist(types, L, min_dist, rng, max_tries=2_000_000):
    coords = []
    placed_types = []
    mind2 = min_dist**2

    for t in types:
        for _ in range(max_tries):
            c = rng.random(3) * L
            ok = True
            for c0 in coords:
                if np.sum((c - c0)**2) < mind2:
                    ok = False
                    break
            if ok:
                coords.append(c)
                placed_types.append(t)
                break
        else:
            raise RuntimeError("Failed to pack atoms: try increasing L_ang or lowering Nw/Nion/min_dist.")
    return np.array(coords), placed_types

# Water geometry (Å): O at origin; two Hs in xz plane
# O-H ~ 0.9572 Å, angle ~ 104.52°
OH = 0.9572
angle = np.deg2rad(104.52)
H1 = np.array([OH, 0.0, 0.0])
H2 = np.array([OH*np.cos(angle), 0.0, OH*np.sin(angle)])

def random_rotation_matrix(rng):
    # Uniform random rotation via quaternion
    u1, u2, u3 = rng.random(3)
    q1 = np.sqrt(1-u1)*np.sin(2*np.pi*u2)
    q2 = np.sqrt(1-u1)*np.cos(2*np.pi*u2)
    q3 = np.sqrt(u1)*np.sin(2*np.pi*u3)
    q4 = np.sqrt(u1)*np.cos(2*np.pi*u3)
    # rotation matrix
    R = np.array([
        [1-2*(q3*q3+q4*q4), 2*(q2*q3-q1*q4),   2*(q2*q4+q1*q3)],
        [2*(q2*q3+q1*q4),   1-2*(q2*q2+q4*q4), 2*(q3*q4-q1*q2)],
        [2*(q2*q4-q1*q3),   2*(q3*q4+q1*q2),   1-2*(q2*q2+q3*q3)]
    ])
    return R

# Create water oxygen "sites" first (then add Hs around them)
O_types = ["O"] * Nw
ion_types = ["Na"] * Nion + ["Cl"] * Nion
all_centers_types = O_types + ion_types

centers, centers_types = place_atoms_with_min_dist(all_centers_types, L_ang, min_dist, rng)

# Expand waters into O/H/H
atom_types = []
atom_coords = []

for i in range(Nw):
    Opos = centers[i]
    R = random_rotation_matrix(rng)
    atom_types.append("O");  atom_coords.append(Opos)
    atom_types.append("H");  atom_coords.append((Opos + R @ H1) % L_ang)
    atom_types.append("H");  atom_coords.append((Opos + R @ H2) % L_ang)

# Add ions
for j in range(2*Nion):
    t = centers_types[Nw + j]
    atom_types.append(t)
    atom_coords.append(centers[Nw + j])

atom_coords = np.array(atom_coords)
len(atom_types), atom_coords.shape


In [ ]:
# Write an XYZ for inspection/visualization
xyz_path = WORK / "nacl_water.xyz"
with xyz_path.open("w") as f:
    f.write(f"{len(atom_types)}\n")
    f.write(f"Lattice="{L_ang} 0 0  0 {L_ang} 0  0 0 {L_ang}" Properties=species:S:1:pos:R:3 pbc="T T T"\n")
    for t, (x, y, z) in zip(atom_types, atom_coords):
        f.write(f"{t:2s} {x:12.6f} {y:12.6f} {z:12.6f}\n")

xyz_path


At this point you can open `nacl_water.xyz` in OVITO / VMD / ASE viewer.

---

## 6) Create PuReMD input files (control + geometry)

Because PuReMD input formats differ slightly between versions, we take this approach:

1. Pick one of the **example control/geo files** printed earlier.
2. Copy it into the working directory.
3. Edit the few key lines (temperature, timestep, steps, file names) programmatically.

If you prefer, you can edit `control` manually after the notebook writes it.


In [ ]:
# Choose templates (adjust if needed)
CONTROL_TEMPLATE = controls[0] if controls else None
GEO_TEMPLATE = geos[0] if geos else None

print("CONTROL_TEMPLATE:", CONTROL_TEMPLATE)
print("GEO_TEMPLATE:", GEO_TEMPLATE)

assert CONTROL_TEMPLATE is not None, "Could not find a PuReMD control template in the repo. Set CONTROL_TEMPLATE manually."
assert GEO_TEMPLATE is not None, "Could not find a PuReMD geometry template in the repo. Set GEO_TEMPLATE manually."


In [ ]:
control_out = WORK / "control"
geo_out = WORK / "geo"

# Copy templates
shutil.copyfile(CONTROL_TEMPLATE, control_out)
shutil.copyfile(GEO_TEMPLATE, geo_out)

print("Wrote:")
print(" -", control_out)
print(" -", geo_out)

print("\n--- control (head) ---")
print("\n".join(control_out.read_text(errors='ignore').splitlines()[:60]))

print("\n--- geo (head) ---")
print("\n".join(geo_out.read_text(errors='ignore').splitlines()[:60]))


### 6a) Overwrite the *geometry* with our NaCl(aq) coordinates

We will attempt to write the `geo` file in a conservative way:

- If the template `geo` appears to be in **XYZ format**, we will write XYZ.
- Otherwise, we will write a simple “n atoms + box + atom lines” style file and you may need to tweak it.

This is the only part of the notebook that may require small edits depending on your PuReMD fork.


In [ ]:
geo_txt = geo_out.read_text(errors="ignore").splitlines()
geo_lower = "\n".join(geo_txt[:20]).lower()

def write_geo_as_xyz(path: Path):
    with path.open("w") as f:
        f.write(f"{len(atom_types)}\n")
        f.write(f"Lattice="{L_ang} 0 0  0 {L_ang} 0  0 0 {L_ang}" Properties=species:S:1:pos:R:3 pbc="T T T"\n")
        for t, (x, y, z) in zip(atom_types, atom_coords):
            f.write(f"{t:2s} {x:12.6f} {y:12.6f} {z:12.6f}\n")

def write_geo_simple(path: Path):
    # Generic fallback: this may need adaptation for your PuReMD version.
    with path.open("w") as f:
        f.write(f"{len(atom_types)}\n")
        f.write(f"# box (A): {L_ang} {L_ang} {L_ang}\n")
        for t, (x, y, z) in zip(atom_types, atom_coords):
            f.write(f"{t:2s} {x:12.6f} {y:12.6f} {z:12.6f}\n")

if "lattice=" in geo_lower or geo_out.suffix.lower() == ".xyz":
    print("Template looks like XYZ / extended XYZ -> writing XYZ geometry.")
    write_geo_as_xyz(geo_out)
else:
    print("Template does not look like XYZ -> writing a simple geometry file (may need small edits).")
    write_geo_simple(geo_out)

print("\n--- geo (new head) ---")
print("\n".join(geo_out.read_text(errors='ignore').splitlines()[:30]))


### 6b) Edit the control file (temperature, pressure, timestep, steps, input/output names)

We will do *minimal* string-based edits:
- replace obvious temperature values with `T_K`
- replace obvious step counts with a modest value
- ensure the control file points to our `geo` and `ffield.reax`

Because PuReMD versions vary, you may need to manually tweak a couple of keywords after this.


In [ ]:
# Workshop-friendly runtime
DT_fs = 0.25
N_STEPS = 20_000          # ~5 ps at 0.25 fs (toy run)

ctrl_lines = control_out.read_text(errors="ignore").splitlines()

def replace_line_if_matches(line, pattern, replacement):
    if re.search(pattern, line, flags=re.IGNORECASE):
        return replacement
    return line

new_lines = []
for line in ctrl_lines:
    # Very conservative: only replace if we see a keyword on the line
    # (You may need to adjust these to match your template.)
    line2 = line

    # File names
    line2 = replace_line_if_matches(line2, r"\bgeo\b|\bgeometry\b|\binput\s*geo\b", f"geo {geo_out.name}")
    line2 = replace_line_if_matches(line2, r"\bffield\b|\breaxff\s*file\b", f"ffield {FFIELD_PATH.name}")

    # Temperature
    if re.search(r"temp|temperature", line2, flags=re.IGNORECASE):
        # keep keyword, replace numeric
        line2 = re.sub(r"([-+]?\d*\.?\d+)", str(T_K), line2, count=1)

    # Pressure (if present)
    if re.search(r"press|pressure", line2, flags=re.IGNORECASE):
        line2 = re.sub(r"([-+]?\d*\.?\d+)", str(P_atm), line2, count=1)

    # Timestep
    if re.search(r"dt\b|timestep", line2, flags=re.IGNORECASE):
        line2 = re.sub(r"([-+]?\d*\.?\d+)", str(DT_fs), line2, count=1)

    # Steps
    if re.search(r"nstep|steps|num_steps", line2, flags=re.IGNORECASE):
        line2 = re.sub(r"(\d+)", str(N_STEPS), line2, count=1)

    new_lines.append(line2)

control_out.write_text("\n".join(new_lines) + "\n")
print("--- control (edited head) ---")
print("\n".join(control_out.read_text(errors='ignore').splitlines()[:80]))


## 7) Run PuReMD

This will run PuReMD in `puremd_nacl_water/` so all outputs land there.

If PuReMD errors out with “could not parse control” or similar:
- scroll up and inspect `control` and `geo`,
- compare against the example files,
- then tweak the relevant keywords.

That debugging step is *also* a nice teaching moment: input decks matter 🙂


In [ ]:
# Sanity checks
assert (WORK / "control").exists()
assert (WORK / "geo").exists()
assert (WORK / "ffield.reax").exists(), "Copy your ReaxFF parameter file to WORK/ffield.reax first."

# Run
cmd = [PUREMD_EXE, "control"]  # many versions accept a single control filename
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=WORK, check=True)


## 8) Inspect outputs

PuReMD output file names vary, but typically you’ll get:
- a log/output text file
- trajectory dumps (often XYZ-like)
- restart files

This cell lists newly created files and prints the tail of any log-like file.


In [ ]:
# List files sorted by modification time
files = sorted(WORK.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
for p in files[:30]:
    print(f"{p.name:30s}  {p.stat().st_size/1024:10.1f} kB")

# Try to find a log-like file
log_candidates = [p for p in files if p.suffix.lower() in {".out", ".log", ".txt"} or "log" in p.name.lower() or "output" in p.name.lower()]
if log_candidates:
    logp = log_candidates[0]
    print("\n--- Tail of", logp.name, "---")
    txt = logp.read_text(errors="ignore").splitlines()
    print("\n".join(txt[-60:]))
else:
    print("\nNo obvious log file found. Open files in the working directory to identify the main output.")


## 9) Optional: quick RDF sanity check (Na–O and Cl–O)

If PuReMD writes an XYZ trajectory (or you convert its trajectory to XYZ), you can compute a basic radial distribution function (RDF) here.

Update `TRAJ_XYZ` to point at your trajectory file.


In [ ]:
TRAJ_XYZ = None  # e.g. WORK / "traj.xyz"

def read_xyz_frames(path: Path):
    lines = path.read_text(errors="ignore").splitlines()
    i = 0
    frames = []
    while i < len(lines):
        n = int(lines[i].strip()); i += 1
        comment = lines[i]; i += 1
        elems = []
        xyz = []
        for _ in range(n):
            parts = lines[i].split(); i += 1
            elems.append(parts[0])
            xyz.append([float(parts[1]), float(parts[2]), float(parts[3])])
        frames.append((np.array(elems), np.array(xyz)))
    return frames

def rdf(frames, species_a, species_b, L, rmax=10.0, nbins=200):
    dr = rmax/nbins
    hist = np.zeros(nbins)
    count_a = 0
    rho_b = None

    for elems, xyz in frames:
        A = np.where(elems == species_a)[0]
        B = np.where(elems == species_b)[0]
        if len(A)==0 or len(B)==0:
            continue
        count_a += len(A)
        # number density of B
        vol = L**3
        rho_b = len(B)/vol

        for ia in A:
            ra = xyz[ia]
            # minimum image distances
            d = xyz[B] - ra
            d -= L*np.round(d/L)
            r = np.linalg.norm(d, axis=1)
            r = r[(r > 1e-6) & (r < rmax)]
            bins = (r/dr).astype(int)
            for b in bins:
                hist[b] += 1

    # Normalize: g(r) = hist / (N_a * rho_b * shell_volume)
    r = (np.arange(nbins) + 0.5)*dr
    shell = 4*np.pi*r*r*dr
    g = hist / (count_a * rho_b * shell)
    return r, g

if TRAJ_XYZ is None:
    print("Set TRAJ_XYZ to an XYZ trajectory file to compute RDFs.")
else:
    frames = read_xyz_frames(Path(TRAJ_XYZ))
    # Use last ~half of frames for a crude “production” window
    frames = frames[len(frames)//2:]
    r, g_nao = rdf(frames, "Na", "O", L_ang, rmax=10.0, nbins=200)
    r, g_clo = rdf(frames, "Cl", "O", L_ang, rmax=10.0, nbins=200)

    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(r, g_nao)
    plt.xlabel("r (Å)")
    plt.ylabel("g_NaO(r)")
    plt.show()

    plt.figure()
    plt.plot(r, g_clo)
    plt.xlabel("r (Å)")
    plt.ylabel("g_ClO(r)")
    plt.show()
